# Health Risk Prediction Models Training
This notebook trains XGBoost Regressors on clinical metrics (age, BMI, blood sugar, BP, smoking) to calculate percentage risks for Heart Disease, Diabetes, and Stroke.

In [ ]:
import os
import pandas as pd
import numpy as np
import pickle
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

# Set paths relative to notebooks directory
BASE_DIR = os.path.dirname(os.getcwd())
DATA_DIR = os.path.join(BASE_DIR, "data")
MODELS_DIR = os.path.join(BASE_DIR, "app", "models")
RAW_DATA_DIR = os.path.join(DATA_DIR, "raw")
PROCESSED_DATA_DIR = os.path.join(DATA_DIR, "processed")

In [ ]:
# Generate / Load clinical dataset
n_samples = 3000
np.random.seed(42)

age = np.random.randint(18, 85, size=n_samples)
bmi = np.random.uniform(15.0, 45.0, size=n_samples)
blood_sugar = np.random.randint(70, 280, size=n_samples)
bp_systolic = np.random.randint(90, 200, size=n_samples)
bp_diastolic = np.random.randint(60, 120, size=n_samples)
smoking = np.random.binomial(1, 0.25, size=n_samples)

noise = np.random.normal(0, 3, size=n_samples)

heart_risk = np.clip(
    5.0 + 0.6 * np.maximum(0, age - 30) + 1.8 * np.maximum(0, bmi - 23) +
    0.4 * np.maximum(0, bp_systolic - 120) + 22.0 * smoking +
    0.1 * np.maximum(0, blood_sugar - 100) + noise, 
    1.0, 99.0
)

diabetes_risk = np.clip(
    3.0 + 0.4 * np.maximum(0, age - 35) + 3.2 * np.maximum(0, bmi - 23) +
    0.45 * np.maximum(0, blood_sugar - 90) + noise,
    1.0, 99.0
)

stroke_risk = np.clip(
    2.0 + 0.5 * np.maximum(0, age - 35) + 0.9 * np.maximum(0, bp_systolic - 120) +
    1.5 * np.maximum(0, bp_diastolic - 80) + 25.0 * smoking +
    0.6 * np.maximum(0, bmi - 24) + noise,
    1.0, 99.0
)

df = pd.DataFrame({
    "age": age, "bmi": bmi, "blood_sugar": blood_sugar,
    "bp_systolic": bp_systolic, "bp_diastolic": bp_diastolic, "smoking": smoking,
    "heart_disease_risk": heart_risk, "diabetes_risk": diabetes_risk, "stroke_risk": stroke_risk
})

df.to_csv(os.path.join(RAW_DATA_DIR, "health_risk_data.csv"), index=False)

In [ ]:
# Train Risk Models
features = ["age", "bmi", "blood_sugar", "bp_systolic", "bp_diastolic", "smoking"]
targets = ["heart_disease_risk", "diabetes_risk", "stroke_risk"]

X = df[features]
y = df[targets]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

models = {}
for target in targets:
    print(f"Training XGBoost regressor for {target}...")
    model = xgb.XGBRegressor(n_estimators=100, max_depth=4, learning_rate=0.08, random_state=42)
    model.fit(X_train, y_train[target])
    preds = model.predict(X_test)
    print(f"{target} MSE: {mean_squared_error(y_test[target], preds):.4f}")
    models[target] = model

In [ ]:
# Save serialized models dict
os.makedirs(MODELS_DIR, exist_ok=True)
with open(os.path.join(MODELS_DIR, "risk_models.pkl"), "wb") as f:
    pickle.dump(models, f)
print("Risk models trained and saved successfully!")